In [3]:
import os
from glob import glob
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, random_split

# ---------------------------------------------------------
# 1. 커스텀 데이터셋 클래스 정의 (가장 핵심적인 변경 사항)
# ---------------------------------------------------------
class LeapGestureDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # 10개의 제스처 클래스 정의
        self.classes = sorted([
            '01_palm', '02_l', '03_fist', '04_fist_moved', '05_thumb',
            '06_index', '07_ok', '08_palm_moved', '09_c', '10_down'
        ])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # 폴더 구조 탐색: root_dir / 피험자(00~09) / 제스처 / 이미지.png
        # 모든 피험자의 이미지를 제스처 기준으로 모읍니다.
        for img_path in glob(os.path.join(root_dir, '*', '*', '*.png')):
            gesture_name = os.path.basename(os.path.dirname(img_path))
            if gesture_name in self.class_to_idx:
                self.image_paths.append(img_path)
                self.labels.append(self.class_to_idx[gesture_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        # [중요] 'RGB'로 열면 1채널 흑백 이미지도 자동으로 3채널로 복제되어 열립니다.
        image = Image.open(img_path).convert('RGB') 
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# ---------------------------------------------------------
# 2. 하드웨어 가속기 및 하이퍼파라미터 설정
# ---------------------------------------------------------
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"현재 구동 중인 디바이스: {device}")

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

# ResNet 입력에 맞춘 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ---------------------------------------------------------
# 3. 데이터 로드 및 Train / Validation 분할
# ---------------------------------------------------------
# ★ 압축을 해제한 LeapGestRecog 폴더의 경로로 변경해 주세요.
data_dir = './leapGestRecog' 

print("데이터를 불러오는 중...")
full_dataset = LeapGestureDataset(root_dir=data_dir, transform=transform)
print(f"총 이미지 개수: {len(full_dataset)}장")

# 전체 데이터의 80%는 학습용, 20%는 검증용으로 나눕니다.
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------
# 4. 모델 설정 (ResNet18 바닥부터 학습)
# ---------------------------------------------------------
model = models.resnet18(weights=None) # 파인튜닝이 아니므로 가중치 None

# 마지막 분류기를 10개 클래스(제스처 개수)에 맞게 변경
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ---------------------------------------------------------
# 5. 본격적인 학습 및 검증 루프
# ---------------------------------------------------------
print("\n--- 학습 시작 ---")
for epoch in range(1, EPOCHS + 1):
    
    # [Train Phase]
    model.train()
    running_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    train_loss = running_loss / len(train_loader)
    train_acc = 100. * train_correct / train_total
    
    # [Validation Phase]
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad(): # 평가 시에는 기울기 계산을 하지 않음 (메모리, 속도 절약)
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100. * val_correct / val_total
    
    print(f"Epoch [{epoch:2d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("--- 학습 완료 ---")

현재 구동 중인 디바이스: mps
데이터를 불러오는 중...
총 이미지 개수: 20000장

--- 학습 시작 ---
Epoch [ 1/10] | Train Loss: 0.1208 | Train Acc: 96.34% | Val Acc: 77.90%
Epoch [ 2/10] | Train Loss: 0.0165 | Train Acc: 99.53% | Val Acc: 99.95%
Epoch [ 3/10] | Train Loss: 0.0106 | Train Acc: 99.71% | Val Acc: 98.58%
Epoch [ 4/10] | Train Loss: 0.0151 | Train Acc: 99.60% | Val Acc: 98.12%
Epoch [ 5/10] | Train Loss: 0.0051 | Train Acc: 99.88% | Val Acc: 100.00%
Epoch [ 6/10] | Train Loss: 0.0069 | Train Acc: 99.81% | Val Acc: 26.23%
Epoch [ 7/10] | Train Loss: 0.0175 | Train Acc: 99.39% | Val Acc: 97.80%
Epoch [ 8/10] | Train Loss: 0.0016 | Train Acc: 99.94% | Val Acc: 99.95%
Epoch [ 9/10] | Train Loss: 0.0003 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch [10/10] | Train Loss: 0.0004 | Train Acc: 99.99% | Val Acc: 100.00%
--- 학습 완료 ---
